# Postselected hard/soft purification at $20\times40$

Runs hard then soft walls at alpha_1=1, then hard then soft at alpha_1=3. Both are deterministic full-postselection trajectories from the maximally mixed state through $4N_y=160$ cycles. Every cycle saves spatial entropy contours, total entropy and the occupation-derived finite-time Lyapunov gap; endpoint spectra and eigenvectors are retained.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import torch

REPORT_ONLY = False
MAX_NEW_CONSTRUCTIONS = None  # all four alpha/wall runs; 1 limits to one pending run
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/19_postselected_hard_soft_n20x40')
LOCAL_BUNDLE_DIR = Path('/content/19_postselected_hard_soft_n20x40')
SCRATCH_ROOT = Path('/content/postselected_hard_soft_n20x40_scratch')
CONFIG = {'sampling_revision': 'postselected_maxmix_hard_soft_alpha1_3_nx20_ny40_s1_4ny_contours_gpu_v2', 'root_seed': 2026092401, 'Nx': 20, 'Ny': 40, 'cycles': 160, 'construction_order': ['hard', 'soft'], 'samples': 1, 'nshell': 1, 'alpha_1': 1.0, 'alpha_1_values': [1.0, 3.0], 'alpha_2': 30.0, 'trial_orbitals': 'X', 'init_mode': 'maxmix', 'sequence': 'raster_y', 'postselect': True, 'postselect_probability': 1.0, 'perfect_correction': False, 'dtype': 'complex128', 'device': 'cuda:0', 'backend': 'local', 'state_representation': 'covariance', 'checkpoint_stride_cycles': 10, 'gpu_memory_minimum_gib': 38.0, 'constructions': {'hard': {'DW': True, 'dw_truncation': True, 'meas_slab_only': True, 'expected_active_modes': 880}, 'soft': {'DW': True, 'dw_truncation': False, 'meas_slab_only': False, 'expected_active_modes': 1600}}}

OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / CONFIG['sampling_revision']

if not torch.cuda.is_available():
    raise RuntimeError('This notebook requires CUDA.')
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in gpu_name or gpu_gib < 38.0:
    raise RuntimeError(f'Expected A100 40-GB-class GPU, got {gpu_name} ({gpu_gib:.2f} GiB)')
print(json.dumps({'config': CONFIG, 'bundle': str(BUNDLE_DRIVE_DIR), 'output': str(OUTPUT_ROOT), 'gpu': gpu_name, 'gpu_GiB': gpu_gib, 'report_only': REPORT_ONLY, 'max_new_constructions': MAX_NEW_CONSTRUCTIONS}, indent=2, sort_keys=True), flush=True)


In [ ]:
import importlib.util
import shutil
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/postselected_hard_soft_n20x40_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
runner_args = ['--config', str(config_path), '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY:
    runner_args.append('--report-only')
if MAX_NEW_CONSTRUCTIONS is not None:
    runner_args.extend(['--max-new-constructions', str(int(MAX_NEW_CONSTRUCTIONS))])
runner_path = LOCAL_BUNDLE_DIR / 'run_campaign.py'
print('[launch in notebook kernel] ' + str(runner_path) + ' ' + ' '.join(runner_args), flush=True)
sys.path.insert(0, str(LOCAL_BUNDLE_DIR))
spec = importlib.util.spec_from_file_location('_postselected_hard_soft_runner', runner_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Could not load staged runner: {runner_path}')
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
returncode = runner.main(runner_args)
if returncode:
    raise RuntimeError(f'Runner exited with status {returncode}')
print('[notebook] hard/soft runner exited successfully', flush=True)


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
